# Bound and drain async work

QuickCart looks up three made-up tickets at once. When one lookup gets stuck, the waiting must stay limited, and a failure must never look like a successful answer.

> **Your goal:** Limit waiting work, clean up properly and keep failures clearly marked as failures.<br>
> **You need:** Functions, exceptions and the validated `Ticket` idea from the previous notebook.<br>
> **Effort:** 45–60 minutes (estimate).<br>
> **You will produce:** Evidence of limited concurrency and timeout cleanup, and your own tested failure records.

## Before executing

Work in your own copy, in a notebook editor running Python 3.11 or later (`asyncio.timeout` needs 3.11). Running it in Colab has not been tested, and hosted notebooks may use a different Python version. You need no extra packages, keys, paid model or real customer data. Keep your work and notes private. Nothing here starts a server or uses a paid API; stop your kernel when you finish.

Start from the cells as supplied, not from an old kernel. Predict, run the cells in order, add the fault described, restore the rule and rerun the same check. Restart before your final in-order run. The notebook starts with all outputs cleared. Note that an unfinished TODO function can sit there while "Run all" still passes; you must call your own function separately to test it.

[Previous: typing and validation](02-typing-and-validation.ipynb) · [Bridge route](README.md) · [Concept explanation](../../study-guides/python-engineering-bridge/README.md)

## Predict before starting work

Three pretend ticket lookups spend their time waiting on the network. Async helps you manage that waiting; it does not make heavy calculations run in parallel. With two slots available, predict:

- How many provider calls can be running at the same time?
- Is the returned list in input order, or in the order the calls finished?
- If one call never finishes, what stops it and waits for it to tidy up?

This example has no effects outside your notebook. Note that cancelling local work would not undo a refund a remote service had already made.

In these notebook cells, use top-level `await`. Calling `asyncio.run()` inside a notebook fails, because the notebook already has an event loop running; a standalone command-line program uses `asyncio.run()` once, at start-up.

In [ ]:
from dataclasses import dataclass


class InputError(ValueError):
    """A declared synthetic ticket contract failed; messages never echo raw payloads."""


@dataclass(frozen=True)
class Ticket:
    ticket_id: str
    amount: int
    tags: tuple[str, ...] = ()

    def __post_init__(self) -> None:
        if not isinstance(self.ticket_id, str) or not self.ticket_id.strip() or len(self.ticket_id) > 64:
            raise InputError('Ticket ID must be nonempty text of at most 64 characters')
        if type(self.amount) is not int or not 0 <= self.amount <= 1_000_000:
            raise InputError('Amount must be an integer in synthetic units, 0–1000000')
        if not isinstance(self.tags, tuple) or len(self.tags) > 10:
            raise InputError('Tags must be a tuple with at most ten values')
        if any(not isinstance(tag, str) or not tag.strip() or len(tag) > 40 for tag in self.tags):
            raise InputError('Tags must be nonempty strings of at most 40 characters')




## Read the collector in stages

The `Ticket` class above is supplied from the validation lesson, so you do not need to rebuild it here. Read the collector below in this order:

1. **Check the limits:** at most 20 tickets, 1–10 calls running at once, and a deadline above zero and no more than 60 seconds.
2. **Hand out slots:** `asyncio.Semaphore` counts the free slots. Each lookup waits for a slot before calling the provider.
3. **Start and collect:** each ticket gets its own task, and `gather` returns the results in input order, even if they finish in a different order.
4. **Set a time limit:** `asyncio.timeout` covers both time spent waiting for a slot and time spent in the call. It relies on the code cooperating: cleanup can run past the deadline, and code that blocks or ignores cancellation can defeat it.
5. **Cancel and wait:** the `finally` block cancels any unfinished lookups and waits for them, so their cleanup runs before the collector returns.

`return_exceptions=True` is used only during that final wait, so that cleanup errors cannot replace the original failure. It does not turn a failed operation into a success. The semaphore limits how many calls are running, not how many tasks exist; here, the separate 20-ticket limit keeps that number small.

In [ ]:
import asyncio
from collections.abc import Awaitable, Callable


async def collect(tickets: list[Ticket], provider: Callable[[Ticket], Awaitable[str]],
                  *, concurrency: int = 2, deadline: float = 0.5) -> list[str]:
    if type(concurrency) is not int or not 1 <= concurrency <= 10 or len(tickets) > 20:
        raise ValueError('At most 20 tickets and 1–10 concurrent calls')
    if type(deadline) not in (int, float) or not 0 < deadline <= 60:
        raise ValueError('Deadline must be positive and at most 60 seconds')
    semaphore = asyncio.Semaphore(concurrency)

    async def one(ticket: Ticket) -> str:
        async with semaphore:
            value = await provider(ticket)
            if not isinstance(value, str):
                raise TypeError('Provider result must be text')
            return value

    tasks: list[asyncio.Task[str]] = []
    try:
        async with asyncio.timeout(deadline):
            tasks = [asyncio.create_task(one(ticket)) for ticket in tickets]
            return list(await asyncio.gather(*tasks))
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)


## Exceptions you already know, with Python differences

Python has no checked exceptions: a function's signature does not list what can fail, so tests and docstrings have to describe that. `raise DomainError(...) from error` works like passing a cause into a Java or C# exception: the original error is kept on `__cause__` for diagnosis, while your message avoids repeating the raw input. Since Python 3.8, `asyncio.CancelledError` is a `BaseException`, not an `Exception`, so `except Exception` does not swallow cancellation, but a bare `except:` or `except BaseException` does. Catch the narrowest error you expect, at the place that can decide what to do about it.

The `parse_amount` example below only demonstrates exception chaining. It deliberately parses text and does not replace the `Ticket` model's stricter rule (a whole number of zero or more). Successfully converting a value does not make it valid business input.

**Converting is not validating:** `int(raw, 10)` also accepts some signs and spaces. The example shows a chained exception, not a full "digits only" rule, and the strict `Ticket` amount rule still applies separately. A safe outer message does not make the original cause safe to log: its text may contain the raw input.

Next, run the normal-provider cell and look at `result`, `peak` and `active`: you should see `A, B, C`, a peak of 2 and no calls still running. The timeout cell then waits on an event that never fires; it should raise `TimeoutError` and set the cleanup flag before control comes back to you. The short sleeps are just for teaching, not a speed benchmark.

In [ ]:
def parse_amount(raw: str) -> int:
    try:
        return int(raw, 10)
    except ValueError as error:
        raise InputError("Amount must be base-10 digits") from error

try:
    parse_amount("12x")
except InputError as error:
    assert isinstance(error.__cause__, ValueError)  # Cause retained for diagnosis.
    assert "12x" not in str(error)  # Message does not echo raw input.
else:
    raise AssertionError("Malformed amount accepted")
assert issubclass(asyncio.CancelledError, BaseException)
assert not issubclass(asyncio.CancelledError, Exception)


In [ ]:
active = 0
peak = 0
async def fake_provider(ticket):
    global active, peak
    active += 1
    peak = max(peak, active)
    try:
        await asyncio.sleep(0.01)
        return ticket.ticket_id
    finally:
        active -= 1
result = await collect([Ticket("A",1),Ticket("B",2),Ticket("C",3)], fake_provider, concurrency=2)
assert result == ["A","B","C"] and peak == 2 and active == 0


In [ ]:
stopped = False
async def stuck_provider(ticket):
    global stopped
    try:
        await asyncio.Event().wait()
    finally:
        stopped = True
try:
    await collect([Ticket("A",1)], stuck_provider, deadline=0.02)
except TimeoutError:
    pass
else:
    raise AssertionError("Expected deadline failure")
assert stopped  # Cleanup happened before control returned.


## Break the concurrency bound, then repair it

In your own copy, replace the `one` helper with this deliberately unsafe version:

```python
async def one(ticket: Ticket) -> str:
    value = await provider(ticket)
    if not isinstance(value, str):
        raise TypeError('Provider result must be text')
    return value
```

Rerun the collector cell, then the normal-provider test cell (which resets the counters). Predict a peak of three instead of two, and watch the assert fail. Put back the original `async with semaphore` block and rerun the same cells. Making the sleep shorter would only hide the bug; it would not restore the limit on how many calls run at once.

## Independent change: preserve failure meaning

Write `collect_with_failures` below. A successful provider call still returns text. An expected provider `RuntimeError` becomes `LookupFailure(ticket_id, "provider_error")`; never copy the raw exception text into that record. Keep the input order, the limits, the result-type check, the deadline and the cleanup. Unexpected errors, such as a provider returning something that is not text, must stay visible. An overall timeout or a cancellation by the caller must be passed on, not turned into a per-ticket failure record.

<details>
<summary>After your attempt: open the hints one by one</summary>

1. Catch only the expected provider error, around the single call, inside its slot.
2. Keep the result-type check separate: a bug or broken contract is not the same as a service outage.
3. Keep the overall timeout and the `finally` cleanup. Never catch `BaseException` just to produce a friendly result.
4. Report answered and failed lookups separately. A failure that was limited and cleaned up is still a failure.

</details>

Once your function is written, add a code cell below it and run:

```python
async def mixed_provider(ticket):
    if ticket.ticket_id == "B":
        raise RuntimeError("Synthetic private detail; do not copy into results")
    return ticket.ticket_id

outcomes = await collect_with_failures(
    [Ticket("A", 1), Ticket("B", 2)], mixed_provider)
assert outcomes == ["A", LookupFailure("B", "provider_error")]

started = asyncio.Event()
cleaned = asyncio.Event()
async def waiting_provider(ticket):
    started.set()
    try:
        await asyncio.Event().wait()
    finally:
        cleaned.set()

caller = asyncio.create_task(collect_with_failures(
    [Ticket("A", 1)], waiting_provider, deadline=5))
await asyncio.wait_for(started.wait(), timeout=1)
caller.cancel()
try:
    await caller
except asyncio.CancelledError:
    pass
else:
    raise AssertionError("Caller cancellation was hidden")
assert cleaned.is_set()
```

Also adapt the existing peak-concurrency and overall-timeout checks so they call your function. Add a case where the provider returns something that is not text (expect `TypeError`) and one with an invalid concurrency value (expect `ValueError`). These examples call your function, so until you write it they fail with `NotImplementedError`; that does not count as finished practice.

In [ ]:
@dataclass(frozen=True)
class LookupFailure:
    ticket_id: str
    reason: str  # A short category such as "provider_error"; never the raw exception text.


async def collect_with_failures(tickets: list[Ticket], provider, *, concurrency: int = 2,
                                deadline: float = 0.5) -> list[str | LookupFailure]:
    # TODO: return the provider text for answered tickets and LookupFailure for an expected
    # provider error such as RuntimeError, in input order.
    # TODO: keep the input bounds, semaphore, deadline and finally-drain behavior from collect.
    # TODO: let asyncio.CancelledError and the overall TimeoutError propagate.
    raise NotImplementedError("Independent extension")

# After implementing, assert one success plus one failure record, peak concurrency, and that
# cancelling the caller still stops a stuck provider. The stub is defined, not exercised.


## Troubleshoot and transfer

| Symptom | Check first | Fix |
|---|---|---|
| "Event loop already running" | Did you paste command-line start-up code into the notebook? | Use top-level `await`. |
| The peak is higher than the number of slots | Is the provider call inside the semaphore? | Restore the slot limit and rerun with the counters reset. |
| The timeout returns before the lookup cleans up | Are unfinished tasks cancelled and then awaited? | Restore the `finally` cleanup and check the cleanup flag. |
| A friendly result appears after the caller cancelled | Is cancellation being caught as a fallback? | Let cancellation pass through and rerun the caller test. |
| Old behaviour continues after your edits | Did you rerun the collector cell? | Restart and run the definitions and tests in order. |

Keep your predictions, the failure and fix, and your own results privately. Clear sensitive output before sharing and stop your session when you finish; nothing here starts a server or costs money.

**Reflect:** which limit controls how many calls start, which controls how long you wait, and what evidence proves cleanup happened? After your selected notebooks, rebuild this boundary in a normal application in the required [package project](python-bridge/README.md). Neither a timeout nor a successful cleanup proves that an external action was undone.

## Check understanding and finish

**Pause and decide:** one lookup succeeds and the other returns a failure record. Did both lookups give useful answers?

<details>
<summary>Compare your reasoning</summary>

No. Record one answer and one failure, separately. Limiting the work and cleaning up show that your controls work; they do not mean the lookup succeeded or that a model answered well.

</details>

Before you call this done, keep:

- Your prediction and the real reference result from a fresh, in-order run.
- The deliberately failing check, your fix and the check passing again.
- Asserts that really call your own function, including invalid input.
- A short explanation of the boundary, and any hints or AI help you used.

**You can now:** limit waiting work, clean up properly and keep failures clearly marked, and prove it with checks rather than a saved green result.

**Next:** open [the next unit](04-tests-and-python-idioms.ipynb) if your route includes it; otherwise go to your next selected unit in the [bridge map](README.md#unit-map-and-estimated-effort). The [package checkpoint](python-bridge/README.md) is still required before the API project. Save your evidence, clear sensitive output before sharing and stop your session.